# Train a Playable-Field Segmentation Model

This notebook trains a one-class YOLO segmentation model that separates the visible playable soccer field from sidelines, benches, tracks, and visually similar grass outside the painted boundaries.

Before running: choose **Runtime > Change runtime type > T4 GPU** (or another GPU). Export the annotations in **YOLOv8 Segmentation** format, ZIP the export, and place it in Google Drive.

## 1. Install dependencies and confirm the GPU

In [ ]:
!pip install -q -U ultralytics pyyaml

import torch
from ultralytics import YOLO

assert torch.cuda.is_available(), 'No GPU detected. Select Runtime > Change runtime type > GPU, then reconnect.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

## 2. Mount Drive and configure paths

Change `DATASET_ZIP` to the location of your Roboflow YOLOv8 Segmentation export. The notebook trains on Colab's local disk for speed and backs results up to Drive.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_PROJECT = Path('/content/drive/MyDrive/soccer-playable-field')
DATASET_ZIP = DRIVE_PROJECT / 'datasets' / 'soccer-playable-field-yolov8.zip'
BACKUP_DIR = DRIVE_PROJECT / 'training_runs'
LOCAL_DATASET_DIR = Path('/content/field_dataset')
LOCAL_RUNS_DIR = Path('/content/field_training_runs')
RUN_NAME = 'playable_field_v1'

BACKUP_DIR.mkdir(parents=True, exist_ok=True)
assert DATASET_ZIP.exists(), f'Dataset ZIP not found: {DATASET_ZIP}'
print('Dataset:', DATASET_ZIP)
print('Backups:', BACKUP_DIR / RUN_NAME)

## 3. Extract and validate the dataset

This locates `data.yaml` even if the ZIP contains an extra top-level folder. It also checks that the dataset contains segmentation polygons rather than detection boxes.

In [ ]:
import shutil
import zipfile
import yaml

if LOCAL_DATASET_DIR.exists():
    shutil.rmtree(LOCAL_DATASET_DIR)
LOCAL_DATASET_DIR.mkdir(parents=True)

with zipfile.ZipFile(DATASET_ZIP) as archive:
    archive.extractall(LOCAL_DATASET_DIR)

yaml_files = list(LOCAL_DATASET_DIR.rglob('data.yaml'))
assert len(yaml_files) == 1, f'Expected one data.yaml, found: {yaml_files}'
DATA_YAML = yaml_files[0]
DATASET_ROOT = DATA_YAML.parent

with DATA_YAML.open() as file:
    dataset_config = yaml.safe_load(file)

print('data.yaml:', DATA_YAML)
print('Classes:', dataset_config.get('names'))
print('Train path:', dataset_config.get('train'))
print('Validation path:', dataset_config.get('val'))
print('Test path:', dataset_config.get('test'))

labels = list(DATASET_ROOT.rglob('labels/*.txt'))
assert labels, 'No YOLO label files found.'
first_values = labels[0].read_text().splitlines()[0].split()
assert len(first_values) > 5, 'Labels appear to be boxes, not segmentation polygons.'
print(f'Found {len(labels)} label files. Segmentation-label check passed.')

## 4. Preview random annotations

Inspect these carefully before training. Polygon edges should follow the playable side of soccer touchlines and endlines.

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt
import numpy as np

image_files = list(DATASET_ROOT.rglob('images/*.jpg')) + list(DATASET_ROOT.rglob('images/*.png'))
sample_images = random.sample(image_files, min(9, len(image_files)))

fig, axes = plt.subplots(3, 3, figsize=(18, 11))
for axis in axes.flat:
    axis.axis('off')
for axis, image_path in zip(axes.flat, sample_images):
    image = cv2.imread(str(image_path))
    h, w = image.shape[:2]
    label_path = Path(str(image_path).replace('/images/', '/labels/')).with_suffix('.txt')
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            values = [float(value) for value in line.split()[1:]]
            points = np.array([(values[i] * w, values[i + 1] * h) for i in range(0, len(values), 2)], np.int32)
            overlay = image.copy()
            cv2.fillPoly(overlay, [points], (0, 180, 0))
            image = cv2.addWeighted(image, 0.65, overlay, 0.35, 0)
            cv2.polylines(image, [points], True, (0, 255, 0), 3)
    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axis.set_title(image_path.name, fontsize=8)
plt.tight_layout()

## 5. Train

Start with the small model. Increase `IMGSZ` or use `yolov8s-seg.pt` only after the pipeline works. `save_period=5` creates checkpoints that can be backed up if Colab disconnects.

In [ ]:
MODEL = 'yolov8n-seg.pt'
EPOCHS = 100
IMGSZ = 960
BATCH = 8

def backup_checkpoint(trainer):
    epoch = trainer.epoch + 1
    if epoch % 5 != 0:
        return
    source = Path(trainer.save_dir) / 'weights' / 'last.pt'
    destination = BACKUP_DIR / RUN_NAME / 'weights' / f'epoch_{epoch:03d}.pt'
    destination.parent.mkdir(parents=True, exist_ok=True)
    if source.exists():
        shutil.copy2(source, destination)
        print(f'Backed up checkpoint: {destination}')

model = YOLO(MODEL)
model.add_callback('on_train_epoch_end', backup_checkpoint)
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=2,
    project=str(LOCAL_RUNS_DIR),
    name=RUN_NAME,
    patience=25,
    save=True,
    save_period=5,
    plots=True,
    degrees=2.0,
    perspective=0.0005,
    scale=0.35,
    fliplr=0.5,
)

## 6. Evaluate and preview predictions

In [ ]:
RUN_DIR = LOCAL_RUNS_DIR / RUN_NAME
BEST_WEIGHTS = RUN_DIR / 'weights' / 'best.pt'
assert BEST_WEIGHTS.exists(), f'Best weights not found: {BEST_WEIGHTS}'

best_model = YOLO(str(BEST_WEIGHTS))
metrics = best_model.val(data=str(DATA_YAML), imgsz=IMGSZ, device=0, plots=True)
print('Mask mAP50:', metrics.seg.map50)
print('Mask mAP50-95:', metrics.seg.map)

preview_value = dataset_config.get('test') or dataset_config['val']
if isinstance(preview_value, list):
    preview_value = preview_value[0]
preview_source = Path(preview_value)
if not preview_source.is_absolute():
    preview_source = (DATASET_ROOT / preview_source).resolve()
if not preview_source.exists():
    candidates = list(DATASET_ROOT.rglob('test/images')) or list(DATASET_ROOT.rglob('valid/images')) or list(DATASET_ROOT.rglob('val/images'))
    assert candidates, f'Could not locate preview images. Configured path: {preview_value}'
    preview_source = candidates[0]
print('Prediction preview source:', preview_source)
best_model.predict(
    source=str(preview_source),
    conf=0.25,
    imgsz=IMGSZ,
    save=True,
    project=str(RUN_DIR),
    name='prediction_preview',
    max_det=1,
)

## 7. Back up the complete run and best weights to Drive

In [ ]:
import shutil

DRIVE_RUN_DIR = BACKUP_DIR / RUN_NAME
if DRIVE_RUN_DIR.exists():
    shutil.rmtree(DRIVE_RUN_DIR)
shutil.copytree(RUN_DIR, DRIVE_RUN_DIR)
shutil.copy2(BEST_WEIGHTS, DRIVE_PROJECT / f'{RUN_NAME}_best.pt')

print('Complete run:', DRIVE_RUN_DIR)
print('Best weights:', DRIVE_PROJECT / f'{RUN_NAME}_best.pt')

## 8. Optional: resume from a checkpoint after a disconnect

Run this instead of the main training cell. Change the checkpoint path if needed.

In [ ]:
# CHECKPOINT = BACKUP_DIR / RUN_NAME / 'weights' / 'last.pt'
# assert CHECKPOINT.exists(), CHECKPOINT
# resumed_model = YOLO(str(CHECKPOINT))
# resumed_model.train(resume=True)